### Import the library

In [1]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

C:\Users\hp\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


NotFoundError: C:\Users\hp\anaconda3\lib\site-packages\tensorflow_decision_forests\tensorflow\ops\inference\inference.so not found

In [2]:
print("TensorFlow v" + tf.__version__)
print("Tensorflow Decision Forests v" + tfdf.__version__)

TensorFlow v2.15.1


NameError: name 'tfdf' is not defined

### Load the dataset

In [ ]:
train_file_path = "../input/house-prices-advances-regression-techniques/train.csv"
dataset_df = pd.read_csv(train_file_path)
print("Full train dataset shape is {}" .format(dataset_df.shape))

In [ ]:
dataset_df.head(3)

### we will drop the ID column as it is not necessary for model training

In [ ]:
dataset_df = dataset_df.drop('ID', axis = 1)
dataset_df.head(3)

### Let's inspect the type of feature columns using the following code:

In [ ]:
dataset_df.info()

### House Price Dsitribution
Now let's look at how the house prices are distributed

In [ ]:
print(dataset_df['SalePrice'].describe())
plt.figure(figsize=(9,8))
sns.distplot(dataset_df['SalePrice'], color='g', bins=100, hist_kws={'alpha': 0.4});

### Numerical data distribution
We will now take a look at how the numerical features are distributed. In order to do this, let us first list all the types of data from our dataset and select only the numerical ones.

In [ ]:
list(set(dataset_df.dtypes.tolist()))

In [ ]:
df_num = dataset_df.select_dtypes(include = ['float64', 'int64'])
df_num.head()

### Now let's plot the distribution for all the numerical features.

In [ ]:
df_num.hist(figsize=(16,20), bins = 50, xlabelsize = 8, ylabelsize=8);

### Prepare the dataset
The dataset contains a mix of numeric, categorical and missing features. TF-DF supports all these ffeature types natively, and no preprocessing is required. This is one advantage of tree-based models, making them a great entry point to Tensorflow and ML.

Now lets split the dataset into training and testing datasets:

In [ ]:
import numpy as np
def split_dataset(dataset, test_ratio=0.30):
    test_indices = np.random.rand(len(dataset)) < test_ratio
    return dataset[~test_indices], dataset[test_indices]

train_ds_pd, valid_ds_pd = split_dataset(dataset_df)
print("{} examples in training, {} examples in testing." .format(len(train_ds_pd), len(valid_ds_pd)))

There's one more step required before we can train the model. We need to convert the dataset from Pandas format(pd.DataFrame) into TensorFlow Datasets format (tf.data.Dataset)

By default the Random Forest Model is configured to train classification tasks. Since this is a regession problem, we will specify the type os the task (tfdf.keras.Task.REGRESSION) as a parameter here.

In [ ]:
label = 'SalePrice'
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_ds_pd, label = label, task = tfdf,keras.Task.REGRESSION)
valid_ds = tfdf.keras.pd_dataframe_to_tf_dataset(valid_ds_pd, label = label, task = tfdf.keras.Task.REGRESSION)

### Select a Model
There are several tree-based models for you to choose from.

I: RandomForestModel
ii: GradientBoostedTreesModel
iii: CartModel
iv: DistributedGradientBoostedTreesModel

To start, I will work with a Random Forest. This is the most well-known of the Decision Forest training algorithms.

A Random Forest is a collection of decision trees, each trained independently on a random subset of the training dataset (sampled with replacement). The algorithm is unique in that it is robust to overfitting and easy to use.

We can list the all the available models in TensorFlow Decision Forests using the following code:

In [ ]:
tfdf.keras.get_all_models()

### How to configure?

TensorFlow Decision Forests provides good defaults for you (e.g. the top ranking hyperparameters on our benchmarks, slightly modified to run in reasonable time). If you would like to configure the learning algorithm, you will find many options you can explore to get the highest possible accuracy

### Create a Random Forest

Today, we will use the defaults to create the Random Forest Model while specifying the task type as tfdf.keras.Task.REGRESSION

In [ ]:
rf = tfdf.keras.RandomForestModel(task = tfdf.keras.Task.REGRESSION)
rf.compile(metrics=["mse"])

### Train the model

I will train model using a one-liner

Not: we may see a warning about Autograph. We can safely ignore this, it will be fixed in the next release.

In [ ]:
rf.fit(x=train_ds)

### Visualize the model

One benefit of tree-based models is that you can easily visualize them. The default number of trees used in the Random Forests is 300. We can select a tree to display below:

In [ ]:
tfdf.model_plotter.plot_model_in_colab(rf, tree_idx=0, max_depth = 3)

### Evaluate the model on the Out of bag (OOB) data and the validation dataset

Before training the dataset we have manually seperated 20% of the dataset for validation named as valid_ds

We can also use Out of bag(OOB) score to validate our RandomForestModel. To train a Random Forest Model, a set of random samples from training set are choosen by the algorithm and the rest of the samples are used to finetune the model.  The subset of data that is not choosen is known as Out of baf data (OOB). OOB score is computed on OOB data.

The training logs show the Root Mean Squared Error(RMSE) evaluated on the out-of-bag dataset according to the number of trees in the model.
Let us plot this

Note: Smaller values are better for the hyperparameter

In [ ]:
import matplotlib.pyplot as plt
logs = rf.make_inspector().training_logs()
plt.plot([log.num_trees for log in logs], [log.evaluation.rmse for log in logs])
plt.xlabel("Number of trees")
plt.ylabel("RMSE (out-of-bag)")
plt.show()

Lets see some general stats on the OOB dataset:

In [ ]:
inspector = rf.make_inspector()
inspectore.evaluation()

Now, lets run an evaluation using the validation dataset.

In [ ]:
evaluation = rf.evaluate(x=valid_ds, return_dict=True)

for name, value in evaluation.items():
    print(f"{name}: {value:.4f})

### Variable importances

Variable importances generally indicate how much a feature contributes of the model predictions or quality. There are several ways to identify important features using TensorFlow Decision Forests. Let us list the available Variable Importances for Decision Trees.

In [ ]:
print(f"Available variable importances:")
for importance in inspector.variable_importances().keys():
    print("\t", importance)

As an example, let us display the important features for the Variable Importance NUM_AS_ROOT.

The larger the importance score for NUM_AS_ROOT, the more impact it has on the outcome of the model.

By default, the list is sorted from the most important to the least. From the output you can infer that the feature at the top of the list is used as the root node in most number of trees in the random forest than any other feature

In [ ]:
inspector.variable_importances()["NUM_AS_ROOT"]

Plotting the variable importances from the inspector using Matplotlib

In [ ]:
plt.figure(figsize=(12, 4))

# Mean decrease in AUC of the class 1 vs the others.
variable_importance_metric = "NUM_AS_ROOT"
variable_importances = inspector.variable_importances()[variable_importance_metric]

# Extract the feature name and importance values.

# variable_importances is a list of <feature, importance> tuples.
feature_names = [vi[0].name for vi in variable_importances]
feature_importances = [vi[1] for vi in variable_importances]
# The feature are ordered in decreasing importance value.
feature_ranks = range(len(feature_names))

bar = plt.barh(feature_ranks, feature_importances, label=[str(x) for x in feature_ranks])
plt.yticks(feature_ranks, feature_names)
plt.gca().invert_yaxis()

# TODO: Replace with "plt.bar_label()" when available
# Label each bar with values
for importance, patch in zip(feature_importances, bar.patches):
    plt.text(patch.get_x() + patch.get_width(), patch.get_y(), f"{importance:.4f, va="top)

plt.xlabel(variable_importance_metric)
plt.title("NUM AS ROOT of the class 1 vs the other s")
plt.tight_layout()
plt.show()

### Submission
Finally predict on the competion test data using the model

In [ ]:
test_file_path = "../input/house-prices-advanced-regression-techniques/test.csv"
test_data = pd.read_csv(test_file_path)
ids = test_data.pop('ID')

test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test_data,
                                               task=tfdf.keras.Task.REGRESSION)
preds = rf.predict(test_ds)
output = pd.DataFrame({'ID':ids,
                      'SalePrice': preds.squeeze()})
output.head()

In [ ]:
sample_submission_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')
sample_submission_df['SalePrice'] = rf.predict(test_ds)
sample_submission_df.to_csv('/kaggle/working/submission.csv', index = False)
sample_submission_df.head()